# 06 — Modern NLP: Hugging Face, Decoder-only LLMs and RAG

> **Notebook 6 of 6 — the capstone** of the *Natural Language Processing from Scratch* series.

After building tokenizers, embeddings and transformer blocks by hand, we now step into
**modern production NLP**: the [Hugging Face Transformers](https://huggingface.co/docs/transformers) ecosystem,
**decoder-only large language models**, prompt-based zero-shot classification and
**Retrieval-Augmented Generation (RAG)**.

## What you will learn
- How `transformers.pipeline` lets you solve real NLP tasks in **3 lines of code**.
- The difference between **encoder** (BERT), **decoder** (GPT) and **encoder-decoder** (T5) families.
- How to **fine-tune** DistilBERT on IMDB with the high-level `Trainer` API.
- **Zero-shot** classification — solving tasks without any training examples.
- A short tour of modern **decoder-only LLMs** (GPT-3.5/4, LLaMA, Mistral).
- A minimal **RAG** pipeline with `sentence-transformers` + FAISS.

## Prerequisites
[Notebook 04 — BERT](04_bert_transformer.ipynb) so you already know what a transformer is.

## How to run this notebook
The Hugging Face models can be heavy. We recommend **Google Colab with a T4 GPU**
(free tier is enough for the whole notebook).

## Estimated runtime
~30–45 minutes on Colab T4.


## 1 — Setup


In [ ]:
# Run this once. Skip if you already have the libraries installed.
!pip install -q transformers datasets accelerate sentence-transformers faiss-cpu


In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


## 2 — First contact with `pipeline`

The `pipeline` function is the highest-level API of Hugging Face.
You pick a **task name**, optionally a **model name**, and the library
takes care of tokenization, model loading and post-processing.

Compare this with **notebook 01**: there we wrote ~50 lines to tokenize, build a
BoW vector and train an SVM. Here we get state-of-the-art sentiment analysis in
**one call**.


In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", device=0 if device == "cuda" else -1)

reviews = [
    "This movie was an absolute masterpiece — gripping plot, brilliant acting.",
    "I fell asleep after 20 minutes. Honestly, save your time.",
    "Not bad, not great. A solid 6/10 for a rainy Sunday.",
]
for r, out in zip(reviews, sentiment(reviews)):
    print(f"{out['label']:<8} ({out['score']:.2f}) — {r}")


### Why this works without training

Behind the scenes, `pipeline("sentiment-analysis")` loads
[`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english)
— a DistilBERT model that was **already fine-tuned** by the community on SST-2
(a movie-review sentiment dataset similar to IMDB).

Compare results on the same 3 reviews:

| Model | Accuracy on IMDB test | Code lines |
|------|----------------------|------------|
| Bag-of-Words + SVM (Notebook 01) | ~82% | ~80 |
| GloVe + Self-Attention (Notebook 03) | ~80% | ~150 |
| BERT fine-tuned (Notebook 04) | ~92% | ~120 |
| **DistilBERT pipeline (this cell)** | **~91%** | **3** |


## 3 — Three model families: encoder, decoder, encoder-decoder

Modern transformer models split into three architectural families.
The choice depends on **what you want to do**:

| Family | Examples | Best for | Direction |
|--------|----------|----------|-----------|
| **Encoder-only** | BERT, RoBERTa, DistilBERT | Classification, NER, embeddings | Bidirectional |
| **Decoder-only** | GPT-2/3/4, LLaMA, Mistral | Text generation, chat, code | Left-to-right |
| **Encoder-decoder** | T5, BART, mT5 | Translation, summarisation | Both |

Let's see one of each in action.


### 3.1 Encoder — BERT for masked language modelling


In [ ]:
mlm = pipeline("fill-mask", model="bert-base-uncased")
results = mlm("The capital of France is [MASK].")
for r in results[:3]:
    print(f"  {r['token_str']:<12} score={r['score']:.3f}")


### 3.2 Decoder — GPT-2 for text generation


In [ ]:
generator = pipeline("text-generation", model="gpt2", device=0 if device == "cuda" else -1)
out = generator(
    "In the year 2030, natural language processing will",
    max_new_tokens=40,
    do_sample=True,
    temperature=0.7,
    pad_token_id=50256,
)
print(out[0]["generated_text"])


### 3.3 Encoder-decoder — T5 for summarisation


In [ ]:
summariser = pipeline("summarization", model="t5-small", device=0 if device == "cuda" else -1)
article = (
    "The Transformer architecture, introduced by Vaswani et al. in 2017, replaced "
    "recurrent and convolutional layers with self-attention. This change unlocked "
    "massively parallel training, allowing models to scale from millions to hundreds "
    "of billions of parameters. Today, every state-of-the-art language model — from "
    "BERT to GPT-4 to LLaMA — is built on this single architectural idea."
)
out = summariser(article, max_length=40, min_length=15, do_sample=False)
print(out[0]["summary_text"])


## 4 — Fine-tuning DistilBERT on IMDB with the `Trainer` API

In **Notebook 04** we wrote a full training loop by hand. That was educational, but
production teams almost always reach for `Trainer`, which handles mixed precision,
gradient accumulation, logging, checkpointing and evaluation.

We will use a **small subset** of IMDB (2k train / 500 val) so this runs in ~5 min on a T4.


In [ ]:
from datasets import load_dataset

imdb = load_dataset("imdb")
train_small = imdb["train"].shuffle(seed=42).select(range(2000))
val_small   = imdb["test"].shuffle(seed=42).select(range(500))
print(train_small)


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

train_tok = train_small.map(tokenize, batched=True)
val_tok   = val_small.map(tokenize, batched=True)


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def accuracy(eval_pred):
    preds, labels = eval_pred
    return {"accuracy": (preds.argmax(-1) == labels).mean()}

args = TrainingArguments(
    output_dir="./distilbert-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=accuracy,
)
trainer.train()


In [ ]:
trainer.evaluate()


### What just happened

In ~30 lines of code we:
1. Loaded a pre-trained DistilBERT model (already trained on Wikipedia + BookCorpus).
2. Fine-tuned only the classification head + a small adjustment of the body.
3. Got **~85–90% accuracy** with just 2k examples and 2 epochs.

This is the magic of **transfer learning**: the heavy lifting (language understanding)
was done once by Google during pre-training. We only adapt the model to our task.


## 5 — Zero-shot classification: no training at all

What if you have **no labelled data**? A zero-shot classifier turns any classification
task into a **textual entailment** problem and solves it with a pre-trained NLI model.

You provide the candidate labels as plain English; the model decides which one best
fits each input. Astonishingly accurate for many real-world use cases.


In [ ]:
zsc = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == "cuda" else -1,
)

texts = [
    "The new graphics card delivers 30% more frames per second.",
    "The Senate approved the budget bill in a 60-40 vote.",
    "She scored two goals in the second half to seal the win.",
]
labels = ["technology", "politics", "sports", "entertainment"]

for t in texts:
    out = zsc(t, candidate_labels=labels)
    top = out["labels"][0]
    score = out["scores"][0]
    print(f"  {top:<13} ({score:.2f}) — {t}")


## 6 — Modern decoder-only LLMs

The most influential models of 2023–2026 — **GPT-3.5/4, LLaMA, Mistral, Claude, Gemini** —
are all **decoder-only** transformers, trained to predict the next token on internet-scale
text. The architectural ideas are the same as the BERT block we built by hand; the
breakthroughs came from **scale**, **data quality** and **alignment** (RLHF, DPO).

Key inference techniques:
- **Prompt engineering** — phrasing your request well dramatically improves results.
- **Few-shot prompting** — show the model 1–5 examples in the prompt.
- **Chain-of-thought** — ask the model to "think step by step".
- **Tool use / function calling** — let the model call external APIs.

Let's demo **few-shot prompting** with GPT-2 (small enough to run for free).
A modern LLM (GPT-4, Claude, LLaMA-70B) would handle this far more reliably,
but the *technique* is identical.


In [ ]:
gen = pipeline("text-generation", model="gpt2", device=0 if device == "cuda" else -1)

few_shot_prompt = """Translate English to French.

English: The book is on the table.
French: Le livre est sur la table.

English: I love natural language processing.
French: J'aime le traitement du langage naturel.

English: The weather is beautiful today.
French:"""

out = gen(
    few_shot_prompt,
    max_new_tokens=20,
    do_sample=False,
    pad_token_id=50256,
)
print(out[0]["generated_text"][len(few_shot_prompt):])


> **Honest disclaimer:** GPT-2 (124M parameters) often gets simple translations wrong.
> A modern LLM with billions of parameters would be near-perfect. The point of this cell
> is to show **the pattern**: examples + new query, and the model continues the sequence.


## 7 — RAG: Retrieval-Augmented Generation

LLMs hallucinate. **Retrieval-Augmented Generation** fixes this by giving the model
relevant **factual context** at query time, retrieved from a vector database.

The recipe:
1. **Embed** all your documents using a sentence encoder.
2. Store them in a **vector index** (FAISS, Pinecone, Chroma, …).
3. At query time, **embed the question**, retrieve the top-K nearest documents.
4. **Prompt the LLM** with: `"Answer using only the context below: <docs> Question: <q>"`.

Below is a minimal end-to-end example with `sentence-transformers` + FAISS.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")

knowledge_base = [
    "BERT was introduced by Google in 2018 and uses bidirectional self-attention.",
    "GPT-2 was released by OpenAI in 2019 and has 1.5 billion parameters in its largest variant.",
    "The Transformer architecture was proposed by Vaswani et al. in the 2017 paper 'Attention Is All You Need'.",
    "LLaMA is a family of open-weight decoder-only language models released by Meta starting in 2023.",
    "Retrieval-Augmented Generation (RAG) combines a retriever with a generator to reduce hallucinations.",
    "Hugging Face is a company and open-source community that hosts the Transformers library and Model Hub.",
]

doc_vectors = encoder.encode(knowledge_base, normalize_embeddings=True)
index = faiss.IndexFlatIP(doc_vectors.shape[1])
index.add(np.array(doc_vectors).astype("float32"))
print(f"Indexed {index.ntotal} documents.")


In [ ]:
def retrieve(question: str, k: int = 2):
    q_vec = encoder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vec, k)
    return [(knowledge_base[i], float(s)) for i, s in zip(ids[0], scores[0])]

question = "Who introduced BERT and when?"
hits = retrieve(question)
for doc, score in hits:
    print(f"  [score={score:.2f}] {doc}")


### Plugging retrieved context into an LLM

In production, you would now build a prompt like:

```
Answer using ONLY the context below. If the answer is not in the context, say "I do not know".

Context:
- BERT was introduced by Google in 2018 and uses bidirectional self-attention.
- The Transformer architecture was proposed by Vaswani et al. in 2017.

Question: Who introduced BERT and when?
Answer:
```

and send it to **GPT-4, Claude, LLaMA-3** or any chat model.
The LLM is forced to ground its answer in the retrieved documents, which dramatically
reduces hallucinations and lets you keep the knowledge base **up-to-date without retraining**.


## 8 — Where to go next

Congratulations! You have travelled from **Bag-of-Words** all the way to **RAG**.
Here are pointers to keep growing:

- 📚 [The Hugging Face NLP Course](https://huggingface.co/course) — free, hands-on, excellent.
- 🛠️ [Andrej Karpathy's *nanoGPT*](https://github.com/karpathy/nanoGPT) — build GPT from scratch in 300 lines.
- 🎓 [Stanford CS224N](https://web.stanford.edu/class/cs224n/) — the canonical NLP course.
- 📰 [Sebastian Raschka's *Build a Large Language Model (from Scratch)*](https://www.manning.com/books/build-a-large-language-model-from-scratch) — end-to-end LLM book.
- 🧪 [LangChain](https://python.langchain.com/) and [LlamaIndex](https://www.llamaindex.ai/) — frameworks for building LLM applications and RAG pipelines.

### Final thought

The field moves fast, but every modern advance — chat models, agents, multimodal
LLMs — is still built on the **self-attention block** we implemented by hand in notebooks
03, 04 and 05. Understanding the fundamentals is what lets you read tomorrow's papers
without fear.

---
**Thanks for reading.** If this repository was useful, please star it on GitHub and feel free to reach out.
